In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

query = "SELECT * FROM retail.fact_demand_features ORDER BY date, product_id, region;"
df_model = pd.read_sql(query, engine)
df_model["date"] = pd.to_datetime(df_model["date"])

print("Dataset loaded successfully. Shape:", df_model.shape)

Dataset loaded successfully. Shape: (39600, 59)


In [5]:
cutoff = (
    df_model["date"].max()
    - pd.Timedelta(weeks=52)
)

train = df_model[
    df_model["date"] < cutoff
].copy()

test = df_model[
    df_model["date"] >= cutoff
].copy()

print(train["date"].min(), train["date"].max())
print(test["date"].min(), test["date"].max())

2022-01-03 00:00:00 2024-10-07 00:00:00
2024-10-14 00:00:00 2025-10-13 00:00:00


Seasonal Naive baseline

In [6]:
test["seasonal_naive"] = test["lag_52"]

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(
    test["demand_units"],
    test["seasonal_naive"]
)

rmse = np.sqrt(
    mean_squared_error(
        test["demand_units"],
        test["seasonal_naive"]
    )
)

print("Seasonal Naive MAE:", mae)
print("Seasonal Naive RMSE:", rmse)

Seasonal Naive MAE: 56.54688679245283
Seasonal Naive RMSE: 108.29663011534394


SARIMAX

In [9]:
weekly = (
    df_model.set_index("date")["demand_units"]
    .resample("W-MON")
    .sum()
)

In [10]:
cutoff = pd.Timestamp(cutoff)

sarima_train = weekly[weekly.index < cutoff]
sarima_test = weekly[weekly.index >= cutoff]

print(f"SARIMA Train Range: {sarima_train.index.min().date()} to {sarima_train.index.max().date()} ({len(sarima_train)} weeks)")
print(f"SARIMA Test Range:  {sarima_test.index.min().date()} to {sarima_test.index.max().date()} ({len(sarima_test)} weeks)")

SARIMA Train Range: 2022-01-03 to 2024-10-07 (145 weeks)
SARIMA Test Range:  2024-10-14 to 2025-10-13 (53 weeks)


In [11]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    sarima_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 0, 52),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_result = model.fit(disp=False)

sarima_forecast = sarima_result.forecast(steps=len(sarima_test))

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

sarima_mae = mean_absolute_error(sarima_test, sarima_forecast)
sarima_rmse = np.sqrt(mean_squared_error(sarima_test, sarima_forecast))

print(f"SARIMAX MAE:  {sarima_mae:.2f}")
print(f"SARIMAX RMSE: {sarima_rmse:.2f}")

SARIMAX MAE:  2274.59
SARIMAX RMSE: 3169.29
